# 🧪 Praktikum Alternatif Bab 09: Prediksi Customer Churn Operator Seluler Menggunakan Ensemble Methods
### Universitas Indraprasta PGRI (Unindra) - Laboratorium Informatika
**Mata Kuliah**: Machine Learning  
**Modul Referensi**: Bab 09  
**Fokus Topik**: Random Forest, Extra Trees, AdaBoost & Gradient Boosting (Studi Kasus: Telekomunikasi)  

---

## 📌 Perbedaan dengan Modul Utama:
> Berbeda dari modul utama (Wine Quality). Kasus Telco Churn mendemonstrasikan kekuatan Bagging vs Boosting dalam mendeteksi pelanggan yang berpotensi berpindah ke kompetitor, dilengkapi Feature Importance.

---

## 🎯 Tujuan Praktikum:
1. Memahami penerapan algoritma pada studi kasus riil yang kontekstual dan mutakhir.
2. Melatih pembersihan, eksplorasi, transformasi, hingga evaluasi metrik kinerja model.
3. Menumbuhkan daya nalar komputasi mahasiswa di luar dataset standar modul kuliah.


## 1. Persiapan Environment & Import Pustaka

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score


## 2. Inisialisasi & Karakterisasi Dataset Alternatif

In [ ]:
# Dataset Simulasi Telco Churn
np.random.seed(77)
n_users = 600

tenure_months = np.random.randint(1, 72, size=n_users)
monthly_charges = np.random.uniform(50000, 450000, size=n_users)
total_calls_support = np.random.poisson(lam=2.5, size=n_users)
contract_monthly = np.random.choice([0, 1], size=n_users, p=[0.4, 0.6]) # 1 = bulan ke bulan
internet_fiber = np.random.choice([0, 1], size=n_users, p=[0.5, 0.5])

# Churn logic: langganan bulanan + tagihan mahal + komplain support tinggi
churn_prob = (contract_monthly * 0.4) + (total_calls_support > 3)*0.35 + (monthly_charges > 300000)*0.2
churn = (churn_prob + np.random.normal(0, 0.1, size=n_users) > 0.55).astype(int)

df_telco = pd.DataFrame({
    'tenure_months': tenure_months,
    'monthly_charges': np.round(monthly_charges, -3),
    'support_calls': total_calls_support,
    'contract_monthly': contract_monthly,
    'internet_fiber': internet_fiber,
    'churn': churn
})

df_telco.to_csv('../../data/raw/alternatif/telco_churn_customers.csv', index=False)
print("Persentase Churn:", df_telco['churn'].mean() * 100, "%")
df_telco.head()


## 3. Implementasi Algoritma & Analisis Kinerja Model

In [ ]:
X = df_telco.drop('churn', axis=1)
y = df_telco['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

models = {
    'Random Forest (Bagging)': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost (Adaptive Boosting)': AdaBoostClassifier(n_estimators=50, random_state=42),
    'Gradient Boosting (Sequential)': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    probs = m.predict_proba(X_test)[:, 1]
    results[name] = {
        'Akurasi': accuracy_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs)
    }

print(pd.DataFrame(results).T)

# Feature Importance dari Random Forest
rf = models['Random Forest (Bagging)']
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
feat_imp.plot(kind='barh', color='teal')
plt.title('Fitur Paling Berpengaruh Terhadap Churn Pelanggan')
plt.xlabel('Gini Importance')
plt.show()


## 4. Tugas Mandiri & Pertanyaan Diskusi Laboratorium:
1. **Analisis Sensitivitas**: Bagaimana perubahan hyperparameter mempengaruhi metrik evaluasi pada dataset ini?
2. **Kesesuaian Algoritma**: Mengapa algoritma yang digunakan cocok untuk karakteristik data di studi kasus ini?
3. **Pertimbangan Bisnis/Etika**: Jika model ini diimplementasikan di dunia nyata, apa risiko kesalahan prediksi (False Positive vs False Negative) yang paling berbahaya?
